# 05 · Feature research and retraining evidence

This appendix reads the exact completed notebook 02 and 03 records. It distinguishes historical baseline results from new training and keeps unsuccessful experiments visible.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import plotly.express as px
from IPython.display import Markdown, display
from march_mania.notebook_support import table, style
from march_mania.publication.workflow import evidence, lineage
from march_mania.advanced_features import candidate_blocks
style()
ROOT = Path.cwd()
if ROOT.name == "notebooks": ROOT = ROOT.parent
FEATURES, FEATURE_RECORD = evidence(ROOT, "feature_store")
MODELS, MODEL_RECORD = evidence(ROOT, "model_comparison")
table(lineage(ROOT))
assert FEATURE_RECORD["summary"]["feature_count"] == MODEL_RECORD["summary"]["feature_count"]
assert FEATURE_RECORD["summary"]["feature_count"] == len(candidate_blocks()["full"])

## Large candidate generation; bounded fitted models

There are 3,106 available candidates, including 2,944 season-local summaries, 26 coach signals and 12 conference-context signals. Per-fold screening, not a global validation-informed filter, caps retained dimensionality at 128. The candidate catalog, full/drop-one ablations and rejection reasons document what was actually tested. Missing women’s coaching sources and rankings remain explicitly unavailable.

In [ ]:
screen = pd.read_csv(FEATURES / "screening_summary.csv")
table(screen.query("block == 'full'")[["Gender", "Season", "model", "candidate_count", "retained_count", "rejected_count"]])
feature_scores = pd.read_csv(FEATURES / "leaderboard.csv")
table(feature_scores.sort_values(["Gender", "macro_season_brier"]).groupby(["Gender", "model"], sort=False).head(4)[["Gender", "block", "model", "brier", "macro_season_brier", "games"]])

## Did retraining change predictions?

The comparison pairs old and revised forecasts by population, season, physical game, route and model. Identical seed-reference scores can be correct; a changed feature fingerprint alone is not evidence of better performance. A negative new-minus-old Brier change is better on these matched development games, not proof of leaderboard performance.

In [ ]:
comparison = pd.read_csv(MODELS / "comparison.csv")
comparison["brier_change"] = comparison.new_brier - comparison.old_brier
table(comparison[["Gender", "block", "model", "games", "old_brier", "new_brier", "brier_change", "max_probability_change"]])

## With and without external ranking information

Men’s ranking logistic, XGBoost and LightGBM candidates are compared with the common/no-Massey model families. Notebook 02 also runs a matched full-versus-without-all-Massey ablation. Women’s and pooled common-feature candidates never consume ranking-derived inputs. Choices and calibration for an outer season use only earlier validation seasons.

In [ ]:
scores = pd.read_csv(MODELS / "leaderboard.csv")
ranked = scores.loc[(scores.Gender == "M") & (scores.block == "M") & scores.model.isin(["rank_logistic_raw", "rank_xgboost_raw", "rank_lightgbm_raw", "logistic_raw", "xgboost_raw", "lightgbm_raw"])]
table(ranked[["model", "brier", "macro_season_brier", "log_loss", "roc_auc", "games"]])
intervals = pd.read_csv(FEATURES / "ablation_intervals.csv")
table(intervals.query("Gender == 'M' and baseline == 'without_massey'"))

## Supported conclusions and boundaries

Use the measured score changes above, not feature count, to judge the experiment. Selection stability and held-out permutation diagnostics appear in notebooks 02 and 03. Large correlated candidate spaces can overfit a small tournament sample even with nested selection. Five development seasons are limited evidence; 2022–2025 was previously consumed and cannot be relabeled as a new holdout. Notebook 04 evaluates the current logistic anchors and retains the earlier frozen final-prediction release separately. The expanded seeded benchmark regresses from 0.198014/0.138599 to 0.198288/0.146881 for men/women, so the enlarged feature bank is not promoted as a universally better forecaster. Submission generation stays opt-in and never uploads to Kaggle.